# <br>

<img src="logo_UNSAM.jpg" align="right" width="180" /> 

<br><br><br>


<h3 align="center">Análisis y Procesamiento de Señales</h3>

<h1 align="center"><b>TS7 -Filtrado Digital de Señales Biomédicas (ECG)</b></h1>
<h3 align="center">Olivia Weihmuller, Luz Romero Marchione y Paloma Ribnikar Diaz</h3>


## Objetivos

El objetivo principal de este trabajo semanal es implementar y analizar técnicas de procesamiento digital no lineales para corregir el movimiento de la línea de base de un registro electrocardiográfico (ECG), y desarrollar un método de detección de complejos QRS.

Para ello, se propone:

- Estimar y eliminar el movimiento de la línea de base mediante filtros de mediana aplicados en dos etapas.
- Implementar una estimación alternativa mediante interpolación con splines cúbicos, utilizando puntos de referencia de la región PQ.
- Comparar ambas estimaciones, evaluando su capacidad para corregir la línea de base y las posibles distorsiones introducidas en el ECG.
- Implementar un filtro adaptado a partir de un patrón QRS conocido y procesar su salida para facilitar la detección de los complejos.
- Evaluar el desempeño del detector mediante su comparación con las detecciones de referencia, calculando la sensibilidad y el valor predictivo positivo (VPP).

## Introducción

En el trabajo semanal anterior se estudió el diseño y la aplicación de filtros digitales lineales para mitigar las interferencias presentes en un registro electrocardiográfico (ECG). En esta práctica se incorporan técnicas de filtrado no lineal, interpolación y detección de eventos, con el objetivo de corregir el movimiento de la línea de base e identificar los complejos QRS.

### Estimación y corrección de la línea de base

La señal registrada puede representarse como $x[n]=s[n]+b[n]+r[n]$, donde $s[n]$ es el ECG de interés, $b[n]$ representa el movimiento de la línea de base y $r[n]$ las restantes interferencias. Para corregirla, se obtiene una estimación $\hat{b}[n]$ que se resta del registro original: $\hat{s}[n]=x[n]-\hat{b}[n]$.

**Filtrado de mediana**

El filtro de mediana es un sistema no lineal que reemplaza cada muestra por la mediana de los valores contenidos en una ventana temporal. Para una ventana impar de longitud $M$, se define:

$$
y[n]=\operatorname{mediana}\left\{x[n-k],\quad k=-\frac{M-1}{2},\ldots,\frac{M-1}{2}\right\}
$$

A diferencia del promedio móvil, presenta mayor robustez frente a valores extremos, lo que permite reducir la influencia de estructuras de gran amplitud y corta duración, como los complejos QRS, al estimar la línea de base. Sin embargo, por su naturaleza no lineal, puede introducir distorsiones en la morfología de la señal.

**Interpolación mediante splines cúbicos**

La interpolación mediante splines cúbicos consiste en construir una función continua a partir de polinomios de tercer grado definidos por tramos, que mantienen continuidad en sus primeras dos derivadas.

En esta práctica, se utiliza para estimar la línea de base a partir de puntos seleccionados en regiones aproximadamente isoeléctricas del ECG, próximas al intervalo PQ. La curva resultante proporciona una estimación suave de la perturbación, aunque su precisión depende de la correcta elección de los puntos de interpolación.

### Detección de complejos QRS mediante filtro adaptado

Un filtro adaptado (*matched filter*) permite identificar regiones de una señal que presentan semejanza con un patrón conocido. Su funcionamiento se basa en la correlación entre la señal y dicho patrón, que puede implementarse mediante una convolución con el patrón invertido temporalmente, $h[n]=p[L-1-n]$.

En este caso, se utiliza un patrón QRS para obtener respuestas de mayor magnitud en regiones con morfología similar. Para facilitar la comparación de las señales, se aplica una normalización:

$$
x_{\mathrm{norm}}[n]=\frac{x[n]-\mu_x}{\sigma_x}
$$

Donde $\mu_x$ y $\sigma_x$ representan la media y la desviación estándar, respectivamente.

### Procesamiento de la señal de detección

Para facilitar la identificación de los máximos se utiliza un **filtro de orden estadístico**, que selecciona un valor de cada ventana según su posición entre las muestras ordenadas.

Mientras que la mediana corresponde al percentil 50, un percentil elevado, como el 90, permite conservar valores próximos a los máximos de la ventana y rellenar pequeñas depresiones de la señal.

Posteriormente, se utiliza `scipy.signal.find_peaks()` para localizar los máximos, estableciendo restricciones como una separación temporal mínima y un umbral de amplitud.

### Evaluación del detector

El desempeño del detector se evalúa mediante una **matriz de confusión**, que permite clasificar las detecciones según su coincidencia con las referencias proporcionadas por la cátedra, considerando una tolerancia temporal determinada.

Se distinguen los verdaderos positivos (TP), correspondientes a QRS correctamente identificados; los falsos positivos (FP), que representan detecciones incorrectas; y los falsos negativos (FN), correspondientes a QRS no detectados.

A partir de estos valores se calculan dos indicadores:

- **Sensibilidad:** proporción de QRS reales correctamente detectados, $\mathrm{Sensibilidad}=\frac{TP}{TP+FN}\cdot100$.
- **Valor predictivo positivo (VPP):** proporción de detecciones que corresponden efectivamente a un QRS, $VPP=\frac{TP}{TP+FP}\cdot100$.

Estas métricas permiten evaluar la capacidad del método para detectar los complejos QRS y distinguirlos de otros eventos presentes en el ECG.